In [17]:
import os, re
import pandas as pd

## Read the atomic number and generate the restraints pair 

### Function

In [30]:
def parse_restraint(restraint):
    try:
        part1, part2 = restraint.split('-')

        def split_part(part):
            match = re.match(r'^(\d+)(.+)', part)
            if match:
                return match.groups()
            else:
                return None, part

        res1, atom1 = split_part(part1)
        res2, atom2 = split_part(part2)

        if res2 is None:
            res2 = res1

        return res1, atom1, res2, atom2

    except Exception as e:
        print(f"Error: {e}")
        return None, None, None, None

### Procession restraint files 

In [32]:
restraint_file = 'restraints.csv'

df_restraints = pd.read_csv(restraint_file)

df_restraints

,nspe_7_1,Intensity
0,6α12-7NH2,weak
1,6α12-7NH1,weak
2,3α12-7NH1,weak
3,1α11-2α2,medium
4,1α12-2α2,medium
5,3α12-2α12,strong
6,4α11-3α11,strong
7,5β3-4α11,medium
8,4α11-5α2,strong
9,4α12-5α2,strong


In [33]:
mol = 'nspe_7_1'

parsed_restraints = []
for i, row in df_restraints.iterrows():
    restraint = row[mol]
    intensity = row['Intensity']

    # Process Restraints 
    res1, atom1, res2, atom2 = parse_restraint(restraint)
    
    
    if None not in (res1, atom1, res2, atom2):
        parsed_restraints.append({
            'res1': res1,
            'atom1': atom1,
            'res2': res2,
            'atom2': atom2,
            'intensity': intensity
        })

# Create the new DataFrame
df_parse_restraints = pd.DataFrame(parsed_restraints)

df_parse_restraints

,res1,atom1,res2,atom2,intensity
0,6,α12,7,NH2,weak
1,6,α12,7,NH1,weak
2,3,α12,7,NH1,weak
3,1,α11,2,α2,medium
4,1,α12,2,α2,medium
5,3,α12,2,α12,strong
6,4,α11,3,α11,strong
7,5,β3,4,α11,medium
8,4,α11,5,α2,strong
9,4,α12,5,α2,strong


In [36]:
df_parse_restraints_renamed = df_parse_restraints
df_parse_restraints_renamed['atom1'] = df_parse_restraints_renamed['atom1'].replace('α12', 'α11')
df_parse_restraints_renamed['atom2'] = df_parse_restraints_renamed['atom2'].replace('α12', 'α11')
df_parse_restraints_renamed['atom1'] = df_parse_restraints_renamed['atom1'].replace('NH2', 'NH1')
df_parse_restraints_renamed['atom2'] = df_parse_restraints_renamed['atom2'].replace('NH2', 'NH1')

df_parse_restraints_renamed

,res1,atom1,res2,atom2,intensity
0,6,α11,7,NH1,weak
1,6,α11,7,NH1,weak
2,3,α11,7,NH1,weak
3,1,α11,2,α2,medium
4,1,α11,2,α2,medium
5,3,α11,2,α11,strong
6,4,α11,3,α11,strong
7,5,β3,4,α11,medium
8,4,α11,5,α2,strong
9,4,α11,5,α2,strong


In [ ]:
# Step 1: Define intensity priority
intensity_priority = {'weak': 0, 'medium': 1, 'strong': 2}
df_parse_restraints_renamed['intensity_score'] = df_parse_restraints_renamed['intensity'].map(intensity_priority)

# Step 2: Sort by priority
df_sorted = df_parse_restraints_renamed.sort_values('intensity_score')

# Step 3: Identify duplicates before dropping
duplicates_mask = df_sorted.duplicated(subset=['res1', 'atom1', 'res2', 'atom2'], keep='first')
df_merged_out = df_sorted[duplicates_mask]

# Step 4: Drop duplicates (keep highest priority)
df_merged = df_sorted.drop_duplicates(subset=['res1', 'atom1', 'res2', 'atom2'], keep='first').drop(columns='intensity_score')

# Optional: print rows that were merged (excluded from final)
print("\n🔁 The following rows were merged (duplicates with lower priority):\n")
print(df_merged_out[['res1', 'atom1', 'res2', 'atom2', 'intensity']])

df_merged = df_merged.reset_index(drop=True)
print("\n✅ Final merged DataFrame with reset index:\n")
print(df_merged)


🔁 The following rows were merged (duplicates with lower priority):

   res1 atom1 res2 atom2 intensity
23    7   α11    7   NH1      weak
1     6   α11    7   NH1      weak
3     1   α11    2    α2    medium
8     4   α11    5    α2    strong
11    5   α11    6    α2    strong

✅ Final merged DataFrame with reset index:

   res1 atom1 res2 atom2 intensity
0     6   α11    7   NH1      weak
1     3   α11    3    α2      weak
2     7   α11    7   NH1      weak
3     3   α11    7   NH1      weak
4     1   α11    2    α2    medium
5     7   α11    7    α2    medium
6     6    β3    6   α11    medium
7     3    β3    3   α11    medium
8     2   α11    2    α2    medium
9     2    β3    2   α11    medium
10    6   α11    7   α11    medium
11    5    β3    4   α11    medium
12    6    β3    5   α11    medium
13    7   α11    6   α11    strong
14    5   α11    6    α2    strong
15    4   α11    5    α2    strong
16    4   α11    3   α11    strong
17    7    β3    7   α11    strong
18    3   α

In [ ]:
cgnr_lookup = {
    'H02': {
        'α11': ['H12', 'H13'],
        'α2': ['H3'],
        'β3': ['H4', 'H5', 'H6']
    },
    'R02': {
        'α11': ['H10', 'H11'],
        'α2': ['H1'],
        'β3': ['H2', 'H3', 'H4']
    }
}

In [46]:
import pandas as pd

# File path setup
itp_path = 'itp'
itp_file = 'nspe_7_1.itp'

start_tag = "[ atoms ]"
block = []
in_block = False

# Step 1: Extract `[ atoms ]` block
with open(f"{itp_path}/{itp_file}", 'r') as file:
    for line in file:
        stripped = line.strip()
        if stripped.startswith('[') and stripped.endswith(']'):
            if stripped.lower() == start_tag:
                in_block = True
                continue
            elif in_block:
                break
        if in_block and stripped and not stripped.startswith(';'):
            block.append(line.strip())

# Step 2: Parse fields from the block
atom_records = []

for line in block:
    # Remove comments
    line = line.split(';')[0].strip()
    if not line:
        continue

    parts = line.split()
    if len(parts) >= 6:
        nr = parts[0]
        atom_type = parts[1]
        resi = parts[2]
        res = parts[3]
        atom = parts[4]
        cgnr = parts[5]
        atom_records.append([nr, atom_type, resi, res, atom, cgnr])

# Step 3: Convert to DataFrame
df_atoms = pd.DataFrame(atom_records, columns=['nr', 'type', 'resi', 'res', 'atom', 'cgnr'])

# Output the result
df_atoms


,nr,type,resi,res,atom,cgnr
0,1,n,1,H02,N,1
1,2,c3,1,H02,C1,2
2,3,c3,1,H02,C2,3
3,4,ca,1,H02,C3,4
4,5,ca,1,H02,C4,5
...,...,...,...,...,...,...
161,162,c,7,R02,C,162
162,163,o,7,R02,O,163
163,164,n,8,NH2,N,164
164,165,hn,8,NH2,H1,165


In [ ]:
for i, row in df_merged.iterrows():
    
    res1_i = row['res1']
    atom1 = row['atom1']
    res2_i = row['res2']
    atom2 = row['atom2']
    intensity = row['intensity']

    print(f"Processing {res1_i, atom1} - {res2_i, atom2}")
    # 1. Find residue name (like 'H02') for res1 and res2
    res1_res_name = df_atoms[df_atoms['resi'] == res1_i]['res'].values[0]
    res2_res_name = df_atoms[df_atoms['resi'] == res2_i]['res'].values[0]

    # 2. Get CGNR labels from the lookup
    cgnr_labels1 = cgnr_lookup.get(res1_res_name, {}).get(atom1, [])
    cgnr_labels2 = cgnr_lookup.get(res2_res_name, {}).get(atom2, [])
    #print(cgnr_labels1)

    # 3. Find matching cgnr numbers in df_atoms
    cgnr_nums_1 = df_atoms[
        (df_atoms['resi'] == res1_i) & 
        (df_atoms['atom'].isin(cgnr_labels1))
    ]['cgnr'].tolist()
    #print(cgnr_nums_1)

    cgnr_nums_2 = df_atoms[
        (df_atoms['resi'] == res2_i) & 
        (df_atoms['atom'].isin(cgnr_labels2))
    ]['cgnr'].tolist()

    # 4. Print or save CGNR bond pairs
    for c1 in cgnr_nums_1:
        for c2 in cgnr_nums_2:
            print(f"CGNR bond: {c1} - {c2} (from {res1_res_name}-{atom1} and {res2_res_name}-{atom2}) with Intensity {intensity}")

    


!!!!"""ArithmeticError

NEED TO ADD THE NH2 as extra

"""""""


Processing ('6', 'α11') - ('7', 'NH1')
Processing ('3', 'α11') - ('3', 'α2')
CGNR bond: 68 - 51 (from R02-α11 and R02-α2) with Intensity weak
CGNR bond: 69 - 51 (from R02-α11 and R02-α2) with Intensity weak
Processing ('7', 'α11') - ('7', 'NH1')
Processing ('3', 'α11') - ('7', 'NH1')
Processing ('1', 'α11') - ('2', 'α2')
CGNR bond: 24 - 28 (from H02-α11 and R02-α2) with Intensity medium
CGNR bond: 25 - 28 (from H02-α11 and R02-α2) with Intensity medium
Processing ('7', 'α11') - ('7', 'α2')
CGNR bond: 160 - 143 (from R02-α11 and R02-α2) with Intensity medium
CGNR bond: 161 - 143 (from R02-α11 and R02-α2) with Intensity medium
Processing ('6', 'β3') - ('6', 'α11')
CGNR bond: 122 - 137 (from R02-β3 and R02-α11) with Intensity medium
CGNR bond: 122 - 138 (from R02-β3 and R02-α11) with Intensity medium
CGNR bond: 123 - 137 (from R02-β3 and R02-α11) with Intensity medium
CGNR bond: 123 - 138 (from R02-β3 and R02-α11) with Intensity medium
CGNR bond: 124 - 137 (from R02-β3 and R02-α11) with I

In [ ]:
itp_path = 'itp'
itp_file = 'nspe_7_1.itp'
restraint_type = 'bonds'


start_tag = f"[ {restraint_type} ]"
block = []
in_block = False

with open(f"{itp_path}/{itp_file}", 'r') as file:
    for line in file:
        stripped = line.strip()
        if stripped.startswith('[') and stripped.endswith(']'):
            if stripped.lower() == start_tag:
                in_block = True
                continue
            elif in_block:
                break  # end of current section
        if in_block:
            block.append(line)

# Print or process the block
for line in block:
    print(line, end='')


;   ai     aj funct   r             k
     1      2   1    1.4620e-01    2.2075e+05 ;      N - C1    
     1     10   1    1.4620e-01    2.2075e+05 ;      N - CA    
     1     13   1    1.0130e-01    4.4124e+05 ;      N - H1    
     1     14   1    1.0130e-01    4.4124e+05 ;      N - H2    
     2      3   1    1.5380e-01    1.9456e+05 ;     C1 - C2    
     2      4   1    1.5160e-01    2.0945e+05 ;     C1 - C3    
     2     15   1    1.0970e-01    3.1455e+05 ;     C1 - H3    
     3     16   1    1.0970e-01    3.1455e+05 ;     C2 - H4    
     3     17   1    1.0970e-01    3.1455e+05 ;     C2 - H5    
     3     18   1    1.0970e-01    3.1455e+05 ;     C2 - H6    
     4      5   1    1.3980e-01    3.1681e+05 ;     C3 - C4    
     4      9   1    1.3980e-01    3.1681e+05 ;     C3 - C8    
     5      6   1    1.3980e-01    3.1681e+05 ;     C4 - C5    
     5     19   1    1.0860e-01    3.3112e+05 ;     C4 - H7    
     6      7   1    1.3980e-01    3.1681e+05 ;     C5 - C6    
  